# Dicionário empírico das tabelas DB2

Este notebook implementa a especificação final do estudo.

## Objetivo

Responder exclusivamente:

- quais colunas existem;
- quais tipos Spark existem;
- quanto cada coluna está preenchida;
- qual a cardinalidade observada;
- quais valores/códigos existem;
- quais textos da **mesma tabela** ajudam a explicar códigos;
- quais associações código ↔ texto são unívocas, não unívocas, parciais ou inexistentes;
- quais códigos permanecem com significado `[PREENCHER]`.

## Fora do escopo

Não há:

- cliente específico;
- Água;
- transferência;
- contraparte;
- conta própria;
- consentimento;
- regra financeira;
- joins entre tabelas;
- categoria original × vigente como regra;
- score/dashboard/recomendação.

## Cinco fontes

1. `DB2GFP.TRAN_RLZD_INST_PCT`
2. `DB2GFP.INF_OPB_CT_CLI`
3. `DB2GFP.CMPT_TRAN_RLZD_CC`
4. `DB2GFP.CTGR_TRAN_OPB`
5. `DB2GFP.GR_CTGR_TRAN`

## Recorte técnico

- `TRAN_RLZD_INST_PCT`: registros com `DT_TRAN` em 2026.
- Demais quatro fontes: tabela completa.

O relatório registra explicitamente o universo efetivamente analisado.

## Cardinalidade

- até 30: valores/frequências;
- 31–100: cardinalidade exata e top 30 para colunas gerais;
- acima de 100: apenas `>100`; top 20 somente quando útil;
- para colunas classificadas como **DOMÍNIO / CÓDIGO**, os códigos até 100 são preservados no dicionário para não eliminar códigos raros;
- `NULL` é apresentado separadamente e não é transformado em código.

## Saída

`estudo_tabelas_resultado.md`

In [ ]:
# ============================================================
# 1. Bootstrap local — padrão do Projeto
# ============================================================
from pathlib import Path

OUTPUT_MD = Path("estudo_tabelas_resultado.md")
spark = None
gerenciador_spark = None

def gravar_falha_bootstrap(etapa, exc):
    OUTPUT_MD.write_text(
        "# Estudo das tabelas\n\n"
        "## Observações técnicas\n\n"
        f"- **EXECUÇÃO INCOMPLETA** — {etapa}: {type(exc).__name__}.\n",
        encoding="utf-8",
    )

try:
    from src.utils.gerenciador_sessao_spark_local import (
        GerenciadorSessaoSpark,
        ler_variavel_ambiente_local,
    )

    ambiente = ler_variavel_ambiente_local("AMBIENTE").upper()
    if ambiente != "MODELAGEM":
        ambiente = "PRODUCAO"

    gerenciador_spark = GerenciadorSessaoSpark(
        nome_sessao="estudo_dicionario_empirico",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            "AMBIENTE": ambiente,
        },
        # Em MODELAGEM, disponibiliza ao Spark remoto as variáveis DB2
        # existentes no desenv.env sem imprimir credenciais.
        nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        driver_memory="12g",
        driver_cores=4,
        executor_memory="12g",
        executor_cores=4,
        num_executors=8,
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "4g",
            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "240",
            "spark.sql.sources.partitionOverwriteMode": "dynamic",
            "spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive": "true",
            "spark.sql.autoBroadcastJoinThreshold": "-1",
            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    print("[OK] Sessão Spark criada.")
except Exception as exc:
    gravar_falha_bootstrap("bootstrap local", exc)
    raise

In [ ]:
# ============================================================
# 2. Abstrações remotas já implementadas no Projeto
# ============================================================
try:
    if spark is None:
        raise RuntimeError("Sessão Spark indisponível.")

    %run ./src/utils/gerenciador_sessao_spark_remoto.ipynb
except Exception as exc:
    gravar_falha_bootstrap("carga do gerenciador Spark remoto", exc)
    raise

In [ ]:
%%spark

# ============================================================
# 3. Cliente DB2 remoto e configuração
# ============================================================
import os
import re
from collections import defaultdict

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    DateType,
    TimestampType,
    BooleanType,
    ByteType,
    ShortType,
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType,
)
from pyspark.storagelevel import StorageLevel

cliente_db2 = criar_cliente_db2_spark(env=dict(os.environ))

TABELAS = [
    "DB2GFP.TRAN_RLZD_INST_PCT",
    "DB2GFP.INF_OPB_CT_CLI",
    "DB2GFP.CMPT_TRAN_RLZD_CC",
    "DB2GFP.CTGR_TRAN_OPB",
    "DB2GFP.GR_CTGR_TRAN",
]

RECORTES = {
    "DB2GFP.TRAN_RLZD_INST_PCT": {
        "universo": "registros de 2026",
        "where": (
            "DT_TRAN >= DATE('2026-01-01') "
            "AND DT_TRAN < DATE('2027-01-01')"
        ),
    },
    "DB2GFP.INF_OPB_CT_CLI": {
        "universo": "tabela completa",
        "where": None,
    },
    "DB2GFP.CMPT_TRAN_RLZD_CC": {
        "universo": "tabela completa",
        "where": None,
    },
    "DB2GFP.CTGR_TRAN_OPB": {
        "universo": "tabela completa",
        "where": None,
    },
    "DB2GFP.GR_CTGR_TRAN": {
        "universo": "tabela completa",
        "where": None,
    },
}

RESULTADO_ESTUDO = {
    "execucao_ok": True,
    "tabelas": {},
    "observacoes_tecnicas": [],
    "dicionario_pendente": [],
    "dicionario_inferido": [],
}

print("[OK] ClientDb2Spark disponível.")

In [ ]:
%%spark

# ============================================================
# 4. Classificação, segurança e perfil de colunas
# ============================================================

DATE_TYPES = (DateType, TimestampType)
NUMERIC_TYPES = (
    ByteType, ShortType, IntegerType, LongType,
    FloatType, DoubleType, DecimalType,
)
INTEGER_TYPES = (ByteType, ShortType, IntegerType, LongType)

TEXT_PREFIXES = (
    "TX_", "NM_", "DS_", "DCR_", "DESC_", "TX_DCR_",
)

DOMAIN_PREFIXES = (
    "CD_", "IN_", "TP_", "TIP_",
)

DOMAIN_TOKENS = (
    "EST", "STS", "FLG", "IND", "NTZ", "MOD", "MOE", "SIT",
)

IDENTIFIER_PATTERNS = [
    r"^CD_CLI$",
    r"^CD_CLI_TITR_CT$",
    r"^NR_TRAN",
    r"^NR_PTC$",
    r"IDFR",
    r"CPF",
    r"CNPJ",
    r"DOCUMENT",
    r"(^|_)DOC($|_)",
    r"PIX",
    r"AGEN",
    r"AGENCIA",
    r"(^|_)NR_CT($|_)",
    r"CONTA",
]

SENSITIVE_PATTERNS = IDENTIFIER_PATTERNS + [
    r"USUAR", r"LOGIN", r"SENHA", r"PASSWORD", r"SECRET", r"TOKEN",
]

def casa_padrao(nome, padroes):
    return any(re.search(p, str(nome), re.I) for p in padroes)

def nome_textual(nome):
    n = str(nome).upper()
    return n.startswith(TEXT_PREFIXES)

def nome_dominio(nome):
    n = str(nome).upper()

    if n.startswith(DOMAIN_PREFIXES):
        return True

    return any(
        re.search(rf"(^|_){token}($|_)", n)
        for token in DOMAIN_TOKENS
    )

def nome_identificador(nome):
    n = str(nome).upper()

    if casa_padrao(n, IDENTIFIER_PATTERNS):
        return True

    # NR_* de alta cardinalidade costuma representar número/identificador.
    if n.startswith("NR_"):
        return True

    return False

def coluna_sensivel(nome):
    return casa_padrao(nome, SENSITIVE_PATTERNS)

def sanitizar_texto_saida(valor):
    if valor is None:
        return None

    texto = str(valor)

    texto = re.sub(
        r"(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b",
        "<EMAIL_OCULTO>",
        texto,
    )
    texto = re.sub(
        r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b",
        "<DOCUMENTO_OCULTO>",
        texto,
    )
    texto = re.sub(
        r"\b\d{2}\.?\d{3}\.?\d{3}/?\d{4}-?\d{2}\b",
        "<DOCUMENTO_OCULTO>",
        texto,
    )
    texto = re.sub(
        r"(?i)\b[0-9a-f]{8}-[0-9a-f]{4}-[1-5][0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}\b",
        "<CHAVE_OCULTA>",
        texto,
    )

    return texto[:1000]

def valor_python(v, sanitizar_texto=False):
    if v is None:
        return None

    if isinstance(v, (bool, int, float)):
        return v

    if hasattr(v, "isoformat"):
        try:
            return v.isoformat()
        except Exception:
            pass

    if sanitizar_texto:
        return sanitizar_texto_saida(v)

    return str(v)

def rows_dict(df, limite, texto_cols=None):
    texto_cols = set(texto_cols or [])
    saida = []

    for row in df.limit(int(limite)).collect():
        item = {}

        for k, v in row.asDict(recursive=True).items():
            item[k] = valor_python(
                v,
                sanitizar_texto=(k in texto_cols),
            )

        saida.append(item)

    return saida

def probe_tabela(tabela):
    # Apenas uma linha para obter schema diretamente pelo Spark/JDBC.
    return cliente_db2.run_select(
        f"SELECT * FROM {tabela} FETCH FIRST 1 ROW ONLY",
        fetchsize=1,
        query_timeout=120,
    )

def carregar_tabela(tabela):
    cfg = RECORTES[tabela]
    probe = probe_tabela(tabela)

    if (
        tabela == "DB2GFP.TRAN_RLZD_INST_PCT"
        and "DT_TRAN" not in probe.columns
    ):
        raise RuntimeError(
            "DT_TRAN não foi observada; o recorte técnico de 2026 não pode ser aplicado."
        )

    sql = f"SELECT * FROM {tabela}"

    if cfg["where"]:
        sql += f" WHERE {cfg['where']}"

    # Para a tabela bilionária, usa JDBC particionado se um identificador
    # inteiro adequado estiver disponível. O bounds é uma consulta técnica única,
    # não uma consulta por coluna.
    if tabela == "DB2GFP.TRAN_RLZD_INST_PCT":
        part_col = None

        for candidato in ["NR_TRAN_INST_PCT", "NR_PTC"]:
            field = next(
                (f for f in probe.schema.fields if f.name == candidato),
                None,
            )

            if (
                field is not None
                and isinstance(field.dataType, INTEGER_TYPES)
            ):
                part_col = candidato
                break

        if part_col:
            bounds_sql = (
                f"SELECT MIN({part_col}) AS MIN_ID, "
                f"MAX({part_col}) AS MAX_ID "
                f"FROM {tabela} "
                f"WHERE {cfg['where']}"
            )

            bounds_df = cliente_db2.run_select(
                bounds_sql,
                fetchsize=1,
                query_timeout=300,
            )

            bounds = bounds_df.collect()[0]
            lower = bounds["MIN_ID"]
            upper = bounds["MAX_ID"]

            if (
                lower is not None
                and upper is not None
                and int(lower) < int(upper)
            ):
                df = cliente_db2.run_select(
                    sql,
                    fetchsize=10000,
                    partition_column=part_col,
                    lower_bound=int(lower),
                    upper_bound=int(upper),
                    num_partitions=32,
                    query_timeout=1200,
                )

                return probe, df, {
                    "estrategia": "JDBC particionado",
                    "coluna_particao": part_col,
                    "particoes": 32,
                }

    df = cliente_db2.run_select(
        sql,
        fetchsize=10000,
        query_timeout=1200,
    )

    return probe, df, {
        "estrategia": "JDBC padrão",
        "coluna_particao": None,
        "particoes": None,
    }

def resumo_base(df):
    """
    Uma única agregação Spark calcula:
    - nulos de todas as colunas;
    - min/max de datas;
    - min/max/média de numéricos;
    - min/média/max de comprimento e strings especiais em textos.
    """
    exprs = []
    meta = {}

    for i, field in enumerate(df.schema.fields):
        c = field.name

        a_null = f"N{i}"
        exprs.append(
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(a_null)
        )
        meta[a_null] = (c, "nulos")

        if isinstance(field.dataType, DATE_TYPES):
            a_min = f"MIN{i}"
            a_max = f"MAX{i}"

            exprs.extend(
                [
                    F.min(c).alias(a_min),
                    F.max(c).alias(a_max),
                ]
            )

            meta[a_min] = (c, "minimo")
            meta[a_max] = (c, "maximo")

        elif isinstance(field.dataType, NUMERIC_TYPES):
            a_min = f"MIN{i}"
            a_max = f"MAX{i}"
            a_avg = f"AVG{i}"

            exprs.extend(
                [
                    F.min(c).alias(a_min),
                    F.max(c).alias(a_max),
                    F.avg(c).alias(a_avg),
                ]
            )

            meta[a_min] = (c, "minimo")
            meta[a_max] = (c, "maximo")
            meta[a_avg] = (c, "media")

        elif isinstance(field.dataType, StringType):
            a_lmin = f"LMIN{i}"
            a_lavg = f"LAVG{i}"
            a_lmax = f"LMAX{i}"
            a_empty = f"EMPTY{i}"
            a_spaces = f"SPACES{i}"

            exprs.extend(
                [
                    F.min(F.length(F.col(c))).alias(a_lmin),
                    F.avg(F.length(F.col(c))).alias(a_lavg),
                    F.max(F.length(F.col(c))).alias(a_lmax),
                    F.sum(
                        F.when(F.col(c) == F.lit(""), 1).otherwise(0)
                    ).alias(a_empty),
                    F.sum(
                        F.when(
                            (F.col(c) != F.lit(""))
                            & (F.trim(F.col(c)) == F.lit("")),
                            1,
                        ).otherwise(0)
                    ).alias(a_spaces),
                ]
            )

            meta[a_lmin] = (c, "comprimento_min")
            meta[a_lavg] = (c, "comprimento_medio")
            meta[a_lmax] = (c, "comprimento_max")
            meta[a_empty] = (c, "strings_vazias")
            meta[a_spaces] = (c, "somente_espacos")

    agg = df.agg(*exprs).first().asDict()

    base = {
        f.name: {
            "nulos": 0,
            "minimo": None,
            "maximo": None,
            "media": None,
            "comprimento_min": None,
            "comprimento_medio": None,
            "comprimento_max": None,
            "strings_vazias": 0,
            "somente_espacos": 0,
        }
        for f in df.schema.fields
    }

    for alias, valor in agg.items():
        coluna, metrica = meta[alias]

        if metrica in ("nulos", "strings_vazias", "somente_espacos"):
            base[coluna][metrica] = int(valor or 0)
        elif metrica in (
            "comprimento_min",
            "comprimento_max",
        ):
            base[coluna][metrica] = (
                int(valor) if valor is not None else None
            )
        elif metrica in ("comprimento_medio", "media"):
            base[coluna][metrica] = (
                round(float(valor), 6)
                if valor is not None
                else None
            )
        else:
            base[coluna][metrica] = valor_python(valor)

    return base

def cardinalidade_ate_101(df, coluna):
    # Não calcula cardinalidade alta exata.
    return (
        df.select(coluna)
        .where(F.col(coluna).isNotNull())
        .dropDuplicates([coluna])
        .limit(101)
        .count()
    )

def classificar_coluna(field, cardinalidade):
    nome = field.name

    if isinstance(field.dataType, DATE_TYPES):
        return "DATA / TIMESTAMP"

    if nome_textual(nome):
        return "TEXTO / DESCRIÇÃO"

    if nome_identificador(nome):
        return "IDENTIFICADOR"

    if nome_dominio(nome) and cardinalidade <= 100:
        return "DOMÍNIO / CÓDIGO"

    if nome_dominio(nome) and cardinalidade > 100:
        # CD_* massivo tende a ser identificador, não domínio.
        return "IDENTIFICADOR"

    if isinstance(field.dataType, BooleanType):
        return "DOMÍNIO / CÓDIGO"

    if isinstance(field.dataType, NUMERIC_TYPES):
        if cardinalidade <= 30:
            return "DOMÍNIO / CÓDIGO"

        return "NUMÉRICO CONTÍNUO"

    if isinstance(field.dataType, StringType):
        return "TEXTO / DESCRIÇÃO"

    return "IDENTIFICADOR"

def frequencias(
    df,
    coluna,
    total,
    limite,
    sanitizar_texto=False,
):
    freq = (
        df.groupBy(coluna)
        .agg(F.count("*").alias("quantidade"))
        .withColumn(
            "percentual",
            F.round(
                F.col("quantidade")
                / F.lit(int(total))
                * F.lit(100.0),
                6,
            )
            if total
            else F.lit(0.0),
        )
        .orderBy(
            F.desc("quantidade"),
            F.col(coluna).asc_nulls_last(),
        )
    )

    return rows_dict(
        freq,
        limite,
        texto_cols=[coluna] if sanitizar_texto else [],
    )

def perfilar_colunas(df, total):
    base = resumo_base(df)
    saida = []

    for field in df.schema.fields:
        c = field.name
        nulos = int(base[c]["nulos"] or 0)
        preenchidos = int(total) - nulos
        card_101 = cardinalidade_ate_101(df, c)

        classificacao = classificar_coluna(
            field,
            card_101,
        )

        card = (
            ">100"
            if card_101 >= 101
            else int(card_101)
        )

        valores = []

        if classificacao == "DOMÍNIO / CÓDIGO":
            # Dicionário: preserva todos os códigos até 100, inclusive raros.
            if card_101 <= 100:
                valores = frequencias(
                    df,
                    c,
                    total,
                    limite=101,  # até 100 não nulos + eventual NULL
                    sanitizar_texto=False,
                )

        elif card_101 <= 30:
            valores = frequencias(
                df,
                c,
                total,
                limite=31,  # até 30 não nulos + eventual NULL
                sanitizar_texto=(
                    classificacao == "TEXTO / DESCRIÇÃO"
                ),
            )

        elif card_101 <= 100:
            valores = frequencias(
                df,
                c,
                total,
                limite=30,
                sanitizar_texto=(
                    classificacao == "TEXTO / DESCRIÇÃO"
                ),
            )

        elif (
            classificacao == "TEXTO / DESCRIÇÃO"
            and not coluna_sensivel(c)
        ):
            valores = frequencias(
                df,
                c,
                total,
                limite=20,
                sanitizar_texto=True,
            )

        saida.append(
            {
                "nome": c,
                "tipo": field.dataType.simpleString().upper(),
                "linhas_analisadas": int(total),
                "preenchidos": preenchidos,
                "nulos": nulos,
                "pct_preenchido": (
                    round(
                        preenchidos / int(total) * 100.0,
                        6,
                    )
                    if total
                    else 0.0
                ),
                "cardinalidade": card,
                "classificacao": classificacao,
                "minimo": base[c]["minimo"],
                "maximo": base[c]["maximo"],
                "media": (
                    None
                    if classificacao == "IDENTIFICADOR"
                    else base[c]["media"]
                ),
                "comprimento_min": base[c]["comprimento_min"],
                "comprimento_medio": base[c]["comprimento_medio"],
                "comprimento_max": base[c]["comprimento_max"],
                "strings_vazias": base[c]["strings_vazias"],
                "somente_espacos": base[c]["somente_espacos"],
                "valores": valores,
                "top_alta_cardinalidade_suprimido": (
                    card == ">100"
                    and classificacao
                    in (
                        "IDENTIFICADOR",
                        "NUMÉRICO CONTÍNUO",
                    )
                ),
            }
        )

    return saida

print("[OK] Classificação e perfil de colunas carregados.")

In [ ]:
%%spark

# ============================================================
# 5. Descoberta automática código ↔ descrição na mesma tabela
# ============================================================

CODE_PREFIXES_STEM = (
    "CD_", "TP_", "TIP_", "IN_",
)

TEXT_PREFIXES_STEM = (
    "TX_DCR_", "DESC_", "DCR_", "TX_", "NM_", "DS_",
)

def remover_prefixo(nome, prefixos):
    n = str(nome).upper()

    for prefixo in sorted(
        prefixos,
        key=len,
        reverse=True,
    ):
        if n.startswith(prefixo):
            return n[len(prefixo):]

    return n

def tokens_nome(nome):
    return [
        t
        for t in re.split(r"[_\W]+", str(nome).upper())
        if t
    ]

def score_par_codigo_texto(col_codigo, col_texto):
    a = remover_prefixo(
        col_codigo,
        CODE_PREFIXES_STEM,
    )
    b = remover_prefixo(
        col_texto,
        TEXT_PREFIXES_STEM,
    )

    if a == b:
        return 100

    if a.endswith(b) or b.endswith(a):
        if min(len(a), len(b)) >= 4:
            return 90

    ta = set(tokens_nome(a))
    tb = set(tokens_nome(b))

    if not ta or not tb:
        return 0

    inter = ta & tb
    uniao = ta | tb
    jaccard = len(inter) / len(uniao)

    if len(inter) >= 2 and jaccard >= 0.66:
        return 80

    if len(inter) >= 2 and jaccard >= 0.50:
        return 70

    return 0

def pares_candidatos(perfil):
    dominios = [
        p["nome"]
        for p in perfil
        if p["classificacao"] == "DOMÍNIO / CÓDIGO"
        and p["cardinalidade"] != ">100"
    ]

    textos = [
        p["nome"]
        for p in perfil
        if p["classificacao"] == "TEXTO / DESCRIÇÃO"
        and nome_textual(p["nome"])
        and not coluna_sensivel(p["nome"])
    ]

    saida = []

    for codigo in dominios:
        candidatos = []

        for texto in textos:
            score = score_par_codigo_texto(
                codigo,
                texto,
            )

            if score >= 70:
                candidatos.append(
                    (score, texto)
                )

        candidatos.sort(
            key=lambda x: (-x[0], x[1])
        )

        for score, texto in candidatos[:3]:
            saida.append(
                {
                    "codigo": codigo,
                    "texto": texto,
                    "score_nome": score,
                }
            )

    return saida

def chave_valor(v):
    if v is None:
        return "__NULL__"

    return (
        type(v).__name__
        + "::"
        + repr(v)
    )

def avaliar_associacao(
    df,
    coluna_codigo,
    coluna_texto,
    cardinalidade_codigo,
    score_nome,
):
    """
    Confirma/rejeita o par pelos dados.
    Não usa o nome como prova.
    """
    pares = (
        df.select(
            coluna_codigo,
            coluna_texto,
        )
        .where(
            F.col(coluna_codigo).isNotNull()
        )
        .groupBy(
            coluna_codigo,
            coluna_texto,
        )
        .agg(
            F.count("*").alias("quantidade")
        )
        .persist(
            StorageLevel.MEMORY_AND_DISK
        )
    )

    # Evita materializar associações absurdamente explosivas no relatório.
    qtd_pares_ate_1001 = (
        pares.select(
            coluna_codigo,
            coluna_texto,
        )
        .limit(1001)
        .count()
    )

    por_codigo = (
        pares.groupBy(coluna_codigo)
        .agg(
            F.sum(
                F.when(
                    F.col(coluna_texto).isNotNull(),
                    1,
                ).otherwise(0)
            ).alias("qt_textos_nao_nulos"),
            F.sum(
                F.when(
                    F.col(coluna_texto).isNull(),
                    F.col("quantidade"),
                ).otherwise(0)
            ).alias("linhas_texto_nulo"),
            F.sum("quantidade").alias("linhas_codigo"),
        )
    )

    stats = por_codigo.agg(
        F.sum(
            F.when(
                F.col("qt_textos_nao_nulos") > 1,
                1,
            ).otherwise(0)
        ).alias("codigos_multiplos_textos"),
        F.sum(
            F.when(
                F.col("qt_textos_nao_nulos") == 1,
                1,
            ).otherwise(0)
        ).alias("codigos_um_texto"),
        F.sum(
            F.when(
                F.col("qt_textos_nao_nulos") == 0,
                1,
            ).otherwise(0)
        ).alias("codigos_sem_texto"),
        F.sum(
            F.when(
                F.col("linhas_texto_nulo") > 0,
                1,
            ).otherwise(0)
        ).alias("codigos_com_nulo"),
    ).first()

    multiplos = int(
        stats["codigos_multiplos_textos"]
        or 0
    )
    um_texto = int(
        stats["codigos_um_texto"]
        or 0
    )
    sem_texto = int(
        stats["codigos_sem_texto"]
        or 0
    )
    com_nulo = int(
        stats["codigos_com_nulo"]
        or 0
    )

    if um_texto == 0 and multiplos == 0:
        situacao = "SEM DESCRIÇÃO ASSOCIADA"

    elif multiplos > 0:
        situacao = "NÃO UNÍVOCA"

    elif (
        sem_texto > 0
        or com_nulo > 0
        or um_texto < int(cardinalidade_codigo)
    ):
        situacao = "PARCIAL"

    else:
        situacao = "UNÍVOCA"

    # Mapeamento por código. Como o domínio tem <=100 códigos,
    # esta saída é necessariamente pequena.
    mapa = {}

    linhas_codigo = (
        pares.orderBy(
            F.col(coluna_codigo).asc_nulls_last(),
            F.desc("quantidade"),
        )
        .limit(1000)
        .collect()
    )

    textos_por_codigo = defaultdict(list)

    for row in linhas_codigo:
        codigo = row[coluna_codigo]
        texto = row[coluna_texto]
        qtd = int(row["quantidade"] or 0)

        textos_por_codigo[
            chave_valor(codigo)
        ].append(
            {
                "codigo": valor_python(codigo),
                "texto": (
                    sanitizar_texto_saida(texto)
                    if texto is not None
                    else None
                ),
                "quantidade": qtd,
            }
        )

    for k, itens in textos_por_codigo.items():
        textos_nao_nulos = [
            x["texto"]
            for x in itens
            if x["texto"] is not None
        ]

        unicos = list(
            dict.fromkeys(textos_nao_nulos)
        )

        mapa[k] = {
            "codigo": itens[0]["codigo"],
            "textos": unicos,
            "texto_inequivoco": (
                unicos[0]
                if len(unicos) == 1
                else None
            ),
            "tem_texto_nulo": any(
                x["texto"] is None
                for x in itens
            ),
        }

    combinacoes = []

    if situacao == "NÃO UNÍVOCA":
        combinacoes = [
            {
                "codigo": item["codigo"],
                "texto": item["texto"],
                "quantidade": item["quantidade"],
            }
            for itens in textos_por_codigo.values()
            for item in itens
        ][:100]

    pares.unpersist()

    return {
        "coluna_codigo": coluna_codigo,
        "coluna_texto": coluna_texto,
        "score_nome": int(score_nome),
        "situacao": situacao,
        "codigos_um_texto": um_texto,
        "codigos_multiplos_textos": multiplos,
        "codigos_sem_texto": sem_texto,
        "codigos_com_nulo": com_nulo,
        "qtd_pares_observados": (
            ">1000"
            if qtd_pares_ate_1001 >= 1001
            else int(qtd_pares_ate_1001)
        ),
        "mapeamento": mapa,
        "combinacoes_nao_univocas": combinacoes,
    }

def descobrir_associacoes(df, perfil):
    perfil_idx = {
        p["nome"]: p
        for p in perfil
    }

    associacoes = []

    for cand in pares_candidatos(perfil):
        codigo = cand["codigo"]
        texto = cand["texto"]
        card = perfil_idx[codigo]["cardinalidade"]

        if card == ">100":
            continue

        associacoes.append(
            avaliar_associacao(
                df,
                codigo,
                texto,
                int(card),
                cand["score_nome"],
            )
        )

    return associacoes

def ranking_associacao(a):
    peso_status = {
        "UNÍVOCA": 4,
        "PARCIAL": 3,
        "NÃO UNÍVOCA": 2,
        "SEM DESCRIÇÃO ASSOCIADA": 1,
    }.get(
        a["situacao"],
        0,
    )

    return (
        peso_status,
        int(a.get("codigos_um_texto", 0)),
        int(a.get("score_nome", 0)),
        -int(a.get("codigos_multiplos_textos", 0)),
    )

def construir_dominio(
    tabela,
    coluna,
    perfil_coluna,
    associacoes,
):
    candidatas = [
        a
        for a in associacoes
        if a["coluna_codigo"] == coluna
    ]

    candidatas.sort(
        key=ranking_associacao,
        reverse=True,
    )

    melhor = (
        candidatas[0]
        if candidatas
        else None
    )

    linhas = []

    for freq in perfil_coluna["valores"]:
        valor = freq.get(coluna)
        qtd = int(freq.get("quantidade", 0))
        pct = float(freq.get("percentual", 0.0))

        if valor is None:
            linhas.append(
                {
                    "codigo": None,
                    "texto_observado": None,
                    "quantidade": qtd,
                    "percentual": pct,
                    "significado": "ausência de valor",
                }
            )
            continue

        texto_obs = None
        significado = "[PREENCHER]"

        if melhor:
            item = melhor["mapeamento"].get(
                chave_valor(valor)
            )

            if item:
                texto_obs = item.get(
                    "texto_inequivoco"
                )

                # Mesmo numa associação PARCIAL, um código específico
                # pode possuir exatamente um texto observado.
                if texto_obs is not None:
                    significado = texto_obs

        linha = {
            "codigo": valor_python(valor),
            "texto_observado": texto_obs,
            "quantidade": qtd,
            "percentual": pct,
            "significado": significado,
        }

        linhas.append(linha)

        if significado == "[PREENCHER]":
            RESULTADO_ESTUDO[
                "dicionario_pendente"
            ].append(
                {
                    "tabela": tabela,
                    "coluna": coluna,
                    "codigo": valor_python(valor),
                    "frequencia": qtd,
                    "texto_associado": (
                        texto_obs
                        if texto_obs is not None
                        else "—"
                    ),
                    "significado": "[PREENCHER]",
                }
            )

    return {
        "coluna": coluna,
        "melhor_associacao": (
            {
                "coluna_texto": melhor["coluna_texto"],
                "situacao": melhor["situacao"],
            }
            if melhor
            else {
                "coluna_texto": None,
                "situacao": "SEM DESCRIÇÃO ASSOCIADA",
            }
        ),
        "valores": linhas,
    }

def consolidar_inferidos(
    tabela,
    associacoes,
):
    for a in associacoes:
        if a["situacao"] not in (
            "UNÍVOCA",
            "PARCIAL",
        ):
            continue

        for item in a["mapeamento"].values():
            texto = item.get(
                "texto_inequivoco"
            )

            if texto is None:
                continue

            RESULTADO_ESTUDO[
                "dicionario_inferido"
            ].append(
                {
                    "tabela": tabela,
                    "coluna_codigo": a[
                        "coluna_codigo"
                    ],
                    "codigo": item["codigo"],
                    "coluna_texto": a[
                        "coluna_texto"
                    ],
                    "texto_observado": texto,
                    "situacao_associacao": a[
                        "situacao"
                    ],
                }
            )

print("[OK] Associação intratabela código ↔ texto carregada.")

In [ ]:
%%spark

# ============================================================
# 6. Perfil completo de uma tabela e execução das cinco fontes
# ============================================================

def periodo_tabela(perfil):
    datas = [
        p
        for p in perfil
        if p["classificacao"] == "DATA / TIMESTAMP"
        and p["minimo"] is not None
        and p["maximo"] is not None
    ]

    if not datas:
        return None

    return [
        {
            "coluna": p["nome"],
            "menor": p["minimo"],
            "maior": p["maximo"],
            "nulos": p["nulos"],
        }
        for p in datas
    ]

def resumo_classificacoes(perfil):
    contagem = defaultdict(int)

    for p in perfil:
        contagem[p["classificacao"]] += 1

    return dict(contagem)

def perfilar_tabela(tabela):
    probe, df, leitura = carregar_tabela(
        tabela
    )

    # Uma única carga de conteúdo é persistida no Spark.
    df = df.persist(
        StorageLevel.MEMORY_AND_DISK
    )

    total = int(df.count())

    perfil = perfilar_colunas(
        df,
        total,
    )

    associacoes = descobrir_associacoes(
        df,
        perfil,
    )

    dominios = []

    for p in perfil:
        if p["classificacao"] != "DOMÍNIO / CÓDIGO":
            continue

        dominios.append(
            construir_dominio(
                tabela,
                p["nome"],
                p,
                associacoes,
            )
        )

    consolidar_inferidos(
        tabela,
        associacoes,
    )

    resumo_class = resumo_classificacoes(
        perfil
    )

    altas = sum(
        1
        for p in perfil
        if p["cardinalidade"] == ">100"
    )

    print(f"[TABELA] {tabela}")
    print(f"Linhas analisadas: {total}")
    print(f"Colunas: {len(probe.columns)}")
    print(
        "Domínios encontrados: "
        f"{resumo_class.get('DOMÍNIO / CÓDIGO', 0)}"
    )
    print(f"Alta cardinalidade: {altas}")
    print(
        "Datas: "
        f"{resumo_class.get('DATA / TIMESTAMP', 0)}"
    )
    print("Concluído.")

    resultado = {
        "tabela": tabela,
        "status": "ANALISADA",
        "universo_analisado": RECORTES[
            tabela
        ]["universo"],
        "linhas": total,
        "colunas": len(
            probe.columns
        ),
        "estrategia_leitura": leitura,
        "periodo": periodo_tabela(perfil),
        "resumo_classificacoes": resumo_class,
        "perfil_colunas": perfil,
        "dominios": dominios,
        "associacoes_codigo_texto": [
            {
                k: v
                for k, v in a.items()
                if k != "mapeamento"
            }
            for a in associacoes
        ],
    }

    df.unpersist()

    return resultado

for tabela in TABELAS:
    try:
        RESULTADO_ESTUDO[
            "tabelas"
        ][tabela] = perfilar_tabela(
            tabela
        )

    except Exception as exc:
        RESULTADO_ESTUDO[
            "execucao_ok"
        ] = False

        codigo = getattr(
            exc,
            "codigo",
            None,
        )

        motivo = (
            f"{type(exc).__name__}"
            + (
                f" [{codigo}]"
                if codigo
                else ""
            )
        )

        RESULTADO_ESTUDO[
            "tabelas"
        ][tabela] = {
            "tabela": tabela,
            "status": "NÃO ANALISADA",
            "universo_analisado": RECORTES[
                tabela
            ]["universo"],
            "motivo_tecnico": motivo,
            "perfil_colunas": [],
            "dominios": [],
            "associacoes_codigo_texto": [],
        }

        RESULTADO_ESTUDO[
            "observacoes_tecnicas"
        ].append(
            {
                "tabela": tabela,
                "observacao": (
                    "Falha parcial: "
                    + motivo
                ),
            }
        )

        print(
            f"[TABELA] {tabela}"
        )
        print(
            "STATUS: NÃO ANALISADA"
        )
        print(
            f"MOTIVO TÉCNICO: {motivo}"
        )

print("[FIM] Perfilamento remoto concluído.")

In [ ]:
# ============================================================
# 7. Markdown final
# ============================================================
import re

def fmt_int(v):
    if v is None:
        return "NÃO DETERMINADO"

    return (
        f"{int(v):,}"
        .replace(",", ".")
    )

def fmt_pct(v):
    if v is None:
        return ""

    return (
        f"{float(v):.6f}"
        .rstrip("0")
        .rstrip(".")
        .replace(".", ",")
        + "%"
    )

def fmt_num(v):
    if v is None:
        return ""

    if isinstance(v, float):
        return (
            f"{v:.6f}"
            .rstrip("0")
            .rstrip(".")
            .replace(".", ",")
        )

    return str(v)

def render_valor(v):
    if v is None:
        return "`NULL`"

    if isinstance(v, str):
        # repr preserva '', espaços e zeros à esquerda.
        valor = repr(v)

        return (
            "`"
            + valor.replace(
                "`",
                "\\`",
            )
            + "`"
        )

    return (
        "`"
        + str(v)
        + "`"
    )

def safe_text(v):
    if v is None:
        return ""

    texto = str(v)

    texto = re.sub(
        r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b",
        "<DOCUMENTO_OCULTO>",
        texto,
    )
    texto = re.sub(
        r"\b\d{2}\.?\d{3}\.?\d{3}/?\d{4}-?\d{2}\b",
        "<DOCUMENTO_OCULTO>",
        texto,
    )

    return (
        texto
        .replace("|", "\\|")
        .replace("\n", " ")
    )

def tabela_md(rows, colunas):
    if not rows:
        return "_Nenhum registro._"

    linhas = [
        "| "
        + " | ".join(
            colunas
        )
        + " |",
        "| "
        + " | ".join(
            "---"
            for _ in colunas
        )
        + " |",
    ]

    for row in rows:
        valores = []

        for c in colunas:
            v = row.get(c, "")

            valores.append(
                safe_text(v)
            )

        linhas.append(
            "| "
            + " | ".join(
                valores
            )
            + " |"
        )

    return "\n".join(
        linhas
    )

try:
    RESULTADO_ESTUDO = (
        spark.get_from_spark(
            "RESULTADO_ESTUDO"
        )
    )
except Exception as exc:
    OUTPUT_MD.write_text(
        "# Estudo das tabelas\n\n"
        "## Observações técnicas\n\n"
        "- **EXECUÇÃO INCOMPLETA** — "
        f"resultado Spark indisponível ({type(exc).__name__}).\n",
        encoding="utf-8",
    )
    raise

linhas = [
    "# Estudo das tabelas",
    "",
    (
        "Produto: dicionário empírico construído "
        "exclusivamente a partir dos valores observados."
    ),
    "",
]

for tabela in [
    "DB2GFP.TRAN_RLZD_INST_PCT",
    "DB2GFP.INF_OPB_CT_CLI",
    "DB2GFP.CMPT_TRAN_RLZD_CC",
    "DB2GFP.CTGR_TRAN_OPB",
    "DB2GFP.GR_CTGR_TRAN",
]:
    t = RESULTADO_ESTUDO[
        "tabelas"
    ].get(
        tabela,
        {},
    )

    linhas += [
        f"# {tabela}",
        "",
        "## Resumo da tabela",
        "",
        (
            "**Universo analisado:** "
            + safe_text(
                t.get(
                    "universo_analisado",
                    "",
                )
            )
        ),
        "",
    ]

    if (
        t.get("status")
        != "ANALISADA"
    ):
        linhas += [
            "**STATUS: NÃO ANALISADA**",
            "",
            (
                "**MOTIVO TÉCNICO:** "
                + safe_text(
                    t.get(
                        "motivo_tecnico",
                        "NÃO DETERMINADO",
                    )
                )
            ),
            "",
        ]
        continue

    rc = t.get(
        "resumo_classificacoes",
        {},
    )

    linhas += [
        (
            f"**Linhas:** "
            f"{fmt_int(t.get('linhas'))}"
        ),
        "",
        (
            f"**Colunas:** "
            f"{fmt_int(t.get('colunas'))}"
        ),
        "",
        (
            "**Colunas domínio:** "
            f"{rc.get('DOMÍNIO / CÓDIGO', 0)}"
        ),
        "",
        (
            "**Colunas identificador:** "
            f"{rc.get('IDENTIFICADOR', 0)}"
        ),
        "",
        (
            "**Colunas texto:** "
            f"{rc.get('TEXTO / DESCRIÇÃO', 0)}"
        ),
        "",
        (
            "**Colunas data:** "
            f"{rc.get('DATA / TIMESTAMP', 0)}"
        ),
        "",
        (
            "**Colunas numéricas:** "
            f"{rc.get('NUMÉRICO CONTÍNUO', 0)}"
        ),
        "",
    ]

    if t.get("periodo"):
        periodos = t[
            "periodo"
        ]

        menor = min(
            (
                str(x["menor"])
                for x in periodos
                if x.get("menor")
                is not None
            ),
            default="",
        )

        maior = max(
            (
                str(x["maior"])
                for x in periodos
                if x.get("maior")
                is not None
            ),
            default="",
        )

        linhas += [
            (
                "**Período observado nas colunas "
                f"temporais:** {menor} a {maior}"
            ),
            "",
        ]

    # Inventário
    linhas += [
        "## Inventário de colunas",
        "",
    ]

    inventario = []

    for p in t.get(
        "perfil_colunas",
        [],
    ):
        inventario.append(
            {
                "Coluna": p["nome"],
                "Tipo": p["tipo"],
                "Preenchidos": fmt_int(
                    p["preenchidos"]
                ),
                "Nulos": fmt_int(
                    p["nulos"]
                ),
                "% Preenchido": fmt_pct(
                    p["pct_preenchido"]
                ),
                "Cardinalidade": p[
                    "cardinalidade"
                ],
                "Classificação": p[
                    "classificacao"
                ],
            }
        )

    linhas += [
        tabela_md(
            inventario,
            [
                "Coluna",
                "Tipo",
                "Preenchidos",
                "Nulos",
                "% Preenchido",
                "Cardinalidade",
                "Classificação",
            ],
        ),
        "",
    ]

    # Domínios
    linhas += [
        "## Domínios e códigos",
        "",
    ]

    if not t.get("dominios"):
        linhas += [
            "_Nenhuma coluna foi classificada "
            "automaticamente como domínio/código._",
            "",
        ]

    for dominio in t.get(
        "dominios",
        [],
    ):
        linhas += [
            f"### {dominio['coluna']}",
            "",
            (
                "**Descrição associada:** "
                + safe_text(
                    dominio[
                        "melhor_associacao"
                    ].get(
                        "coluna_texto"
                    )
                    or "não encontrada"
                )
            ),
            "",
            (
                "**Situação:** "
                + safe_text(
                    dominio[
                        "melhor_associacao"
                    ].get(
                        "situacao"
                    )
                )
            ),
            "",
        ]

        rows_dom = []

        for item in dominio[
            "valores"
        ]:
            rows_dom.append(
                {
                    "Código": render_valor(
                        item["codigo"]
                    ),
                    "Texto observado": (
                        render_valor(
                            item[
                                "texto_observado"
                            ]
                        )
                        if item.get(
                            "texto_observado"
                        )
                        is not None
                        else "—"
                    ),
                    "Quantidade": fmt_int(
                        item["quantidade"]
                    ),
                    "%": fmt_pct(
                        item["percentual"]
                    ),
                    "Significado": (
                        safe_text(
                            item[
                                "significado"
                            ]
                        )
                    ),
                }
            )

        linhas += [
            tabela_md(
                rows_dom,
                [
                    "Código",
                    "Texto observado",
                    "Quantidade",
                    "%",
                    "Significado",
                ],
            ),
            "",
        ]

    # Associações
    linhas += [
        "## Associações código ↔ texto",
        "",
    ]

    assoc_rows = []

    for a in t.get(
        "associacoes_codigo_texto",
        [],
    ):
        assoc_rows.append(
            {
                "Código": a[
                    "coluna_codigo"
                ],
                "Coluna textual": a[
                    "coluna_texto"
                ],
                "Situação": a[
                    "situacao"
                ],
                "Códigos com 1 texto": a[
                    "codigos_um_texto"
                ],
                "Códigos com >1 texto": a[
                    "codigos_multiplos_textos"
                ],
                "Códigos sem texto": a[
                    "codigos_sem_texto"
                ],
            }
        )

    linhas += [
        tabela_md(
            assoc_rows,
            [
                "Código",
                "Coluna textual",
                "Situação",
                "Códigos com 1 texto",
                "Códigos com >1 texto",
                "Códigos sem texto",
            ],
        ),
        "",
    ]

    for a in t.get(
        "associacoes_codigo_texto",
        [],
    ):
        if (
            a.get("situacao")
            != "NÃO UNÍVOCA"
        ):
            continue

        linhas += [
            (
                f"### Associação não unívoca: "
                f"`{a['coluna_codigo']}` ↔ "
                f"`{a['coluna_texto']}`"
            ),
            "",
        ]

        combos = []

        for x in a.get(
            "combinacoes_nao_univocas",
            [],
        ):
            combos.append(
                {
                    "Código": render_valor(
                        x["codigo"]
                    ),
                    "Texto": (
                        render_valor(
                            x["texto"]
                        )
                        if x.get("texto")
                        is not None
                        else "`NULL`"
                    ),
                    "Quantidade": fmt_int(
                        x["quantidade"]
                    ),
                }
            )

        linhas += [
            tabela_md(
                combos,
                [
                    "Código",
                    "Texto",
                    "Quantidade",
                ],
            ),
            "",
        ]

    # Datas
    linhas += [
        "## Datas",
        "",
    ]

    datas = []

    for p in t.get(
        "perfil_colunas",
        [],
    ):
        if (
            p["classificacao"]
            != "DATA / TIMESTAMP"
        ):
            continue

        datas.append(
            {
                "Coluna": p["nome"],
                "Menor valor": p[
                    "minimo"
                ]
                or "",
                "Maior valor": p[
                    "maximo"
                ]
                or "",
                "Nulos": fmt_int(
                    p["nulos"]
                ),
            }
        )

    linhas += [
        tabela_md(
            datas,
            [
                "Coluna",
                "Menor valor",
                "Maior valor",
                "Nulos",
            ],
        ),
        "",
    ]

    # IDs / alta cardinalidade
    linhas += [
        "## Identificadores / alta cardinalidade",
        "",
    ]

    altas = []

    for p in t.get(
        "perfil_colunas",
        [],
    ):
        if not (
            p["classificacao"]
            == "IDENTIFICADOR"
            or p["cardinalidade"]
            == ">100"
        ):
            continue

        observacao = []

        if (
            p["classificacao"]
            == "IDENTIFICADOR"
        ):
            observacao.append(
                "valores individuais não expostos"
            )

        if (
            p["cardinalidade"]
            == ">100"
        ):
            observacao.append(
                "cardinalidade exata não calculada"
            )

        altas.append(
            {
                "Coluna": p["nome"],
                "Cardinalidade": p[
                    "cardinalidade"
                ],
                "Preenchimento": fmt_pct(
                    p[
                        "pct_preenchido"
                    ]
                ),
                "Observação": "; ".join(
                    observacao
                ),
            }
        )

    linhas += [
        tabela_md(
            altas,
            [
                "Coluna",
                "Cardinalidade",
                "Preenchimento",
                "Observação",
            ],
        ),
        "",
    ]

    # Estatísticas textuais / numéricas
    linhas += [
        "## Estatísticas técnicas complementares",
        "",
    ]

    extras = []

    for p in t.get(
        "perfil_colunas",
        [],
    ):
        if (
            p["classificacao"]
            == "TEXTO / DESCRIÇÃO"
        ):
            extras.append(
                {
                    "Coluna": p["nome"],
                    "Classe": p[
                        "classificacao"
                    ],
                    "Mínimo": "",
                    "Máximo": "",
                    "Média": "",
                    "Compr. mín.": p[
                        "comprimento_min"
                    ],
                    "Compr. médio": fmt_num(
                        p[
                            "comprimento_medio"
                        ]
                    ),
                    "Compr. máx.": p[
                        "comprimento_max"
                    ],
                    "Strings vazias": fmt_int(
                        p[
                            "strings_vazias"
                        ]
                    ),
                    "Somente espaços": fmt_int(
                        p[
                            "somente_espacos"
                        ]
                    ),
                }
            )

        elif (
            p["classificacao"]
            == "NUMÉRICO CONTÍNUO"
        ):
            extras.append(
                {
                    "Coluna": p["nome"],
                    "Classe": p[
                        "classificacao"
                    ],
                    "Mínimo": fmt_num(
                        p["minimo"]
                    ),
                    "Máximo": fmt_num(
                        p["maximo"]
                    ),
                    "Média": fmt_num(
                        p["media"]
                    ),
                    "Compr. mín.": "",
                    "Compr. médio": "",
                    "Compr. máx.": "",
                    "Strings vazias": "",
                    "Somente espaços": "",
                }
            )

    linhas += [
        tabela_md(
            extras,
            [
                "Coluna",
                "Classe",
                "Mínimo",
                "Máximo",
                "Média",
                "Compr. mín.",
                "Compr. médio",
                "Compr. máx.",
                "Strings vazias",
                "Somente espaços",
            ],
        ),
        "",
    ]

    # Pontos a preencher
    linhas += [
        "## Pontos para preencher manualmente",
        "",
    ]

    pendentes_tabela = [
        x
        for x in RESULTADO_ESTUDO.get(
            "dicionario_pendente",
            [],
        )
        if x["tabela"] == tabela
    ]

    if not pendentes_tabela:
        linhas += [
            "_Nenhum código pendente nesta tabela._",
            "",
        ]

    else:
        por_coluna = {}

        for x in pendentes_tabela:
            por_coluna.setdefault(
                x["coluna"],
                [],
            ).append(x)

        for coluna, itens in por_coluna.items():
            linhas.append(
                f"- `{coluna}`"
            )

            for x in itens:
                linhas.append(
                    "  - "
                    + render_valor(
                        x["codigo"]
                    )
                    + " → `[PREENCHER]`"
                )

        linhas.append("")

# Seções finais
linhas += [
    "# Dicionário pendente de significado",
    "",
]

pend_rows = []

for x in RESULTADO_ESTUDO.get(
    "dicionario_pendente",
    [],
):
    pend_rows.append(
        {
            "Tabela": x["tabela"],
            "Coluna": x["coluna"],
            "Código": render_valor(
                x["codigo"]
            ),
            "Frequência": fmt_int(
                x["frequencia"]
            ),
            "Texto associado": safe_text(
                x["texto_associado"]
            ),
            "Significado": "[PREENCHER]",
        }
    )

linhas += [
    tabela_md(
        pend_rows,
        [
            "Tabela",
            "Coluna",
            "Código",
            "Frequência",
            "Texto associado",
            "Significado",
        ],
    ),
    "",
    "# Dicionário inferido diretamente da própria tabela",
    "",
]

inf_rows = []

for x in RESULTADO_ESTUDO.get(
    "dicionario_inferido",
    [],
):
    inf_rows.append(
        {
            "Tabela": x["tabela"],
            "Coluna código": x[
                "coluna_codigo"
            ],
            "Código": render_valor(
                x["codigo"]
            ),
            "Coluna texto": x[
                "coluna_texto"
            ],
            "Texto observado": safe_text(
                x["texto_observado"]
            ),
            "Situação": x[
                "situacao_associacao"
            ],
        }
    )

linhas += [
    tabela_md(
        inf_rows,
        [
            "Tabela",
            "Coluna código",
            "Código",
            "Coluna texto",
            "Texto observado",
            "Situação",
        ],
    ),
    "",
    "# Observações técnicas",
    "",
    "- As cinco tabelas foram estudadas isoladamente.",
    "- Não foram feitos joins entre tabelas.",
    "- `TRAN_RLZD_INST_PCT` usa somente o recorte técnico de 2026 por `DT_TRAN`.",
    "- As demais tabelas foram configuradas para análise integral.",
    "- Nenhuma coluna é ignorada silenciosamente: todas aparecem no inventário.",
    "- A cardinalidade alta é interrompida logicamente ao confirmar mais de 100 valores.",
    "- Associações código ↔ texto são apenas intratabela e são confirmadas pelo comportamento observado dos dados.",
    "- Nenhum significado externo foi acrescentado.",
]

for obs in RESULTADO_ESTUDO.get(
    "observacoes_tecnicas",
    [],
):
    linhas.append(
        "- "
        + safe_text(
            obs.get("tabela")
        )
        + ": "
        + safe_text(
            obs.get("observacao")
        )
    )

if not RESULTADO_ESTUDO.get(
    "execucao_ok"
):
    linhas += [
        "",
        (
            "**EXECUÇÃO INCOMPLETA:** "
            "uma ou mais tabelas não puderam ser analisadas. "
            "As demais seções foram preservadas."
        ),
    ]

OUTPUT_MD.write_text(
    "\n".join(linhas).strip()
    + "\n",
    encoding="utf-8",
)

print(
    f"[OK] Gerado: {OUTPUT_MD}"
)

## Produto final

Execute o notebook e revise somente:

`estudo_tabelas_resultado.md`

O arquivo apresenta o inventário completo de colunas, domínios, associações código ↔ texto confirmadas nos próprios dados, códigos pendentes de significado e significados inferidos exclusivamente da própria tabela.